In [5]:
import anndata as ad
import numpy as np
import pandas as pd

In [6]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

In [7]:
import pickle

In [8]:
import pubchempy as pcp
from tqdm import tqdm

In [10]:
def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [11]:
df_pert = pd.read_pickle('../../../lpm_style/lpm_style_embeddings_epoch_5/df_pert.pkl')

In [12]:
tahoe_sci_op3_updated = pd.read_pickle('../../tahoe_sci_op3_updated.pkl')

In [13]:
sci = tahoe_sci_op3_updated[tahoe_sci_op3_updated['dataset'] == 'sciplex3'].reset_index(drop=True)

In [14]:
sci = sci.drop(columns=['cmap_name', 'symbol', 'code', 'symbol_', 'LPM_emb'])

In [15]:
sci['pubchem_cid'] = sci['pubchem_cid'].astype(str)

In [16]:
sci = sci.merge(df_pert, left_on='pubchem_cid', right_on='symbol', how='left')

In [17]:
sci['pubchem_cid'] = sci['pubchem_cid'].astype(int)

In [19]:
sci[sci['lpm_style_embeddings'].isna()]

,perturbagen,pubchem_cid,smiles,dataset,ECFP:2,original_pert_name,symbol,code,lpm_style_embeddings
14,DMSO,679,CS(=O)C,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",control,NaN,NaN,NaN
37,Aurora,86222,CCOC(=O)C(CC1=CC(=C(C=C1Cl)F)N2C(=O)N(C(=N2)C)...,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Aurora A Inhibitor I,NaN,NaN,NaN
39,Divalproex,23663956,CCCC(CCC)C(=O)O.CCCC(CCC)C(=O)[O-].[Na+],sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Divalproex Sodium,NaN,NaN,NaN
41,SRT3025,46245047,COCCCC1=C(N=C(S1)C2=CC=CC=C2)C(=O)NC3=CC=CC=C3...,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",SRT3025 HCl,NaN,NaN,NaN
59,AR-42,6918848,CC(C)C(C1=CC=CC=C1)C(=O)NC2=CC=C(C=C2)C(=O)NO,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",AR-42,NaN,NaN,NaN
74,sodium atom,5360545,[Na],sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Sodium Phenylbutyrate,NaN,NaN,NaN
80,tranylcypromine,5530,C1C(C1N)C2=CC=CC=C2,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Tranylcypromine (2-PCPA) HCl,NaN,NaN,NaN
132,epothilone,10838895,CC1CCCC2C(O2)CC(OC(=O)CC(C(C(=O)C(C1O)C)(C)C)O...,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Epothilone A,NaN,NaN,NaN
141,INO-1001,11272610,CS(=O)(=O)O.C1COCCN1CCCNS(=O)(=O)C2=CC3=C(C=C2...,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",INO-1001 (3-Aminobenzamide),NaN,NaN,NaN
151,Cyclocytidine,25051,C1=CN2C3C(C(C(O3)CO)O)OC2=NC1=N,sciplex3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Cyclocytidine HCl,NaN,NaN,NaN


In [20]:
sci.to_pickle("sci_lpm_style_embeddings_epoch5.pkl")